# ToysAD-8K Opaque Package Smoke

File: `03_toysad8k_opaque_package_smoke.ipynb`

This notebook verifies the fixed 20-scene integration package used by the first model experiments. The package is built locally from the trusted source records, then published privately with opaque scene and object IDs. Kaggle independently verifies the published package and records the evidence.

The package contains 20 scenes and all 20 views per scene. It does not contain labels, categories, roles, asset groups, source identifiers, masks, source paths, or evaluator records.


## Acceptance contract

This is a coverage-focused integration cohort, not an unbiased performance sample. It is accepted only when:

- the local build passes its trusted-source checks;
- the private Kaggle copy matches the accepted package hashes;
- exactly 20 scenes and 400 colour PNG views are packaged;
- the output contains only the approved manifest, record, checksum file, and media files;
- a second build reuses all 400 files without changing package hashes;
- the saved tables, figures, run record, and artifact hashes are complete; and
- the corresponding ClearML task contains only safe aggregate evidence.


In [ ]:
from hashlib import sha256
from html import escape
import importlib.util
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time

from IPython.display import HTML, display
from PIL import Image


In [ ]:
NOTEBOOK_NAME = "03_toysad8k_opaque_package_smoke.ipynb"


def find_code_root(start: Path) -> Path:
    candidates = []
    configured = os.getenv("VPI_CODE_ROOT")
    if configured:
        candidates.append(Path(configured).expanduser())
    candidates.extend((start, *start.parents))
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        for mounted in sorted(kaggle_input.iterdir()):
            candidates.extend((mounted, mounted / "code"))
        for marker in kaggle_input.rglob("model_safe_package.py"):
            if marker.parts[-3:] == (
                "visual_peer_inspector",
                "data",
                "model_safe_package.py",
            ):
                candidates.append(marker.parents[3])
    for candidate in candidates:
        marker = candidate / "src" / "visual_peer_inspector" / "data" / "model_safe_package.py"
        if marker.is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate the code snapshot. Set VPI_CODE_ROOT to the directory containing src/visual_peer_inspector."
    )


def choose_directory(variable: str, default: Path, candidates: list[Path], marker: str) -> Path:
    configured = os.getenv(variable)
    ordered = [Path(configured).expanduser()] if configured else []
    ordered.extend([default, *candidates])
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        ordered.extend(path.parent for path in kaggle_input.rglob(marker))
    for candidate in ordered:
        if candidate.is_dir() and (candidate / marker).exists():
            return candidate.resolve()
    raise FileNotFoundError(
        f"Could not locate {variable}. Set it to a directory containing {marker}."
    )


def prepare_clearml_tracking(execution_environment: str) -> str:
    default_mode = "online" if execution_environment == "kaggle" else "disabled"
    tracking_mode = os.getenv("VPI_TRACKING_MODE", default_mode).lower()
    os.environ["VPI_TRACKING_MODE"] = tracking_mode
    os.environ.setdefault("CLEARML_LOG_ENVIRONMENT", "")
    if tracking_mode == "disabled":
        return tracking_mode
    if importlib.util.find_spec("clearml") is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "clearml==2.1.10"]
        )
    if tracking_mode == "online" and execution_environment == "kaggle":
        from kaggle_secrets import UserSecretsClient

        secret_client = UserSecretsClient()
        for name in ("CLEARML_API_ACCESS_KEY", "CLEARML_API_SECRET_KEY"):
            try:
                value = secret_client.get_secret(name)
            except Exception as error:
                raise RuntimeError(f"Kaggle Secret {name} is missing or disabled.") from error
            if not value:
                raise RuntimeError(f"Kaggle Secret {name} is empty.")
            os.environ[name] = value
        os.environ.setdefault("CLEARML_API_HOST", "https://api.clear.ml")
        os.environ.setdefault("CLEARML_WEB_HOST", "https://app.clear.ml")
        os.environ.setdefault("CLEARML_FILES_HOST", "https://files.clear.ml")
    return tracking_mode


def verify_snapshot_record(code_root: Path, snapshot_path: Path) -> dict:
    from visual_peer_inspector.verification import verify_code_snapshot_record

    return verify_code_snapshot_record(code_root, snapshot_path)


def read_provenance(code_root: Path, notebook_name: str) -> dict:
    notebook_path = code_root / "notebooks" / notebook_name
    if not notebook_path.is_file():
        raise FileNotFoundError(f"Canonical notebook source is missing: {notebook_name}")
    snapshot_candidates = [
        Path(os.environ["VPI_SNAPSHOT_RECORD"]).expanduser()
        if os.getenv("VPI_SNAPSHOT_RECORD") else None,
        code_root / "snapshot_record.json",
        code_root / ".local" / "kaggle_code_snapshot" / "snapshot_record.json",
    ]
    snapshot_path = next(
        (path for path in snapshot_candidates if path is not None and path.is_file()),
        None,
    )
    if snapshot_path is not None:
        snapshot = verify_snapshot_record(code_root, snapshot_path)
        git_commit = snapshot["git_commit"]
        git_dirty = bool(snapshot["git_dirty"])
        snapshot_sha256 = snapshot["snapshot_sha256"]
        platform_versions = dict(snapshot.get("platform_versions", {}))
        precommit_candidate = bool(snapshot.get("precommit_candidate", False))
    else:
        repository_root = code_root.parent
        git_commit = subprocess.check_output(
            ["git", "-C", str(repository_root), "rev-parse", "HEAD"],
            text=True,
        ).strip()
        git_dirty = bool(
            subprocess.check_output(
                ["git", "-C", str(repository_root), "status", "--porcelain"],
                text=True,
            ).strip()
        )
        snapshot_sha256 = "local-working-tree"
        platform_versions = {}
        precommit_candidate = git_dirty
    return {
        "git_commit": git_commit,
        "git_dirty": git_dirty,
        "snapshot_sha256": snapshot_sha256,
        "precommit_candidate": precommit_candidate,
        "platform_versions": platform_versions,
        "notebook_sha256": sha256(notebook_path.read_bytes()).hexdigest(),
    }


def display_rows(rows: list[dict], columns: list[str], limit: int | None = None) -> None:
    shown = rows if limit is None else rows[:limit]
    header = "".join(f"<th>{escape(column)}</th>" for column in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(row.get(column, '')))}</td>" for column in columns) + "</tr>"
        for row in shown
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))


## Runtime and trusted inputs

Local execution builds from the ignored trusted inputs. Kaggle receives only the code snapshot and the private opaque package. Restricted source maps, labels, source identifiers, and evaluator records are not attached. ClearML is enabled by default on Kaggle and reads its credentials from private Kaggle Secrets.


In [ ]:
EXECUTION_ENVIRONMENT = os.getenv(
    "VPI_EXECUTION_ENVIRONMENT",
    "kaggle" if Path("/kaggle/working").is_dir() else "local",
).lower()
if EXECUTION_ENVIRONMENT not in {"local", "kaggle"}:
    raise ValueError("VPI_EXECUTION_ENVIRONMENT must be local or kaggle.")
CODE_ROOT = find_code_root(Path.cwd().resolve())
sys.path.insert(0, str(CODE_ROOT / "src"))

INSTALL_REQUIREMENTS = os.getenv(
    "VPI_INSTALL_REQUIREMENTS",
    "1" if EXECUTION_ENVIRONMENT == "kaggle" else "0",
) == "1"
if INSTALL_REQUIREMENTS:
    requirement = CODE_ROOT / "requirements" / "opaque-package-lock.txt"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirement)])

TRACKING_MODE = prepare_clearml_tracking(EXECUTION_ENVIRONMENT)

from visual_peer_inspector.data.model_safe_package import (
    SMOKE20_COHORT_ID,
    build_smoke20_rgb_package,
    collect_model_input_evidence,
    file_sha256,
    verify_model_input_package,
)
from visual_peer_inspector.evidence import (
    save_bar_chart,
    save_histogram,
    save_image_grid,
    write_csv_rows,
)
from visual_peer_inspector.tracking import ExperimentTracker

kaggle_input = Path("/kaggle/input")
mounted_roots = sorted(kaggle_input.iterdir()) if kaggle_input.is_dir() else []
local_selection = CODE_ROOT / ".local" / "toysad" / "selection" / "toysad8k-train-2000-v2"
if EXECUTION_ENVIRONMENT == "kaggle":
    PUBLISHED_INPUT_ROOT = choose_directory(
        "VPI_MODEL_INPUT_ROOT",
        CODE_ROOT / ".local" / "toysad" / "runs" / "opaque-package-smoke20" / "model-input",
        [root for mounted in mounted_roots for root in (mounted, mounted / "model-input")],
        "package_record.json",
    )
    PACKAGE_OPERATION = "verify_published_package"
else:
    SELECTION_ROOT = choose_directory(
        "VPI_SELECTION_ROOT",
        local_selection,
        [],
        "neutral_corpus_manifest.jsonl",
    )
    DEVELOPMENT_ROOT = choose_directory(
        "VPI_DEVELOPMENT_ROOT",
        local_selection.with_name(local_selection.name + "-development"),
        [],
        "development_source_map.jsonl",
    )
    ADMINISTRATIVE_ROOT = choose_directory(
        "VPI_ADMINISTRATIVE_ROOT",
        local_selection.with_name(local_selection.name + "-administrative-vault"),
        [],
        "required_source_members.jsonl",
    )
    SOURCE_ROOT = choose_directory(
        "VPI_SOURCE_ROOT",
        CODE_ROOT / ".local" / "toysad" / "kaggle" / "toysad8k-full-private-v1",
        [],
        "package-checksums.sha256",
    )
    PACKAGE_OPERATION = "build_and_verify"
default_output = (
    Path("/kaggle/working/toysad8k_opaque_package_smoke")
    if EXECUTION_ENVIRONMENT == "kaggle"
    else CODE_ROOT / ".local" / "toysad" / "runs" / "opaque-package-smoke20"
)
OUTPUT_ROOT = Path(os.getenv("VPI_OUTPUT_ROOT", str(default_output))).expanduser().resolve()
REQUIRE_FRESH_OUTPUT = os.getenv(
    "VPI_REQUIRE_FRESH_OUTPUT",
    "1" if EXECUTION_ENVIRONMENT == "kaggle" else "0",
) == "1"
PROVENANCE = read_provenance(CODE_ROOT, NOTEBOOK_NAME)
if (
    EXECUTION_ENVIRONMENT == "kaggle"
    and PROVENANCE["git_dirty"]
    and not PROVENANCE["precommit_candidate"]
):
    raise RuntimeError(
        "A dirty Kaggle snapshot must be marked as a pre-commit candidate."
    )
version_variables = (
    {
        "kaggle_notebook_version_03": "VPI_KAGGLE_NOTEBOOK_VERSION_03",
        "code_dataset_version": "VPI_CODE_DATASET_VERSION",
        "model_input_dataset_version": "VPI_MODEL_INPUT_DATASET_VERSION",
    }
    if EXECUTION_ENVIRONMENT == "kaggle"
    else {}
)
PLATFORM_VERSIONS = {
    key: os.getenv(variable) or PROVENANCE["platform_versions"].get(key)
    for key, variable in version_variables.items()
}
missing_versions = [key for key, value in PLATFORM_VERSIONS.items() if not value]
if EXECUTION_ENVIRONMENT == "kaggle" and missing_versions:
    raise RuntimeError(
        f"Kaggle version metadata is missing: {missing_versions}"
    )
PLATFORM_VERSIONS = {
    key: value or "local" for key, value in PLATFORM_VERSIONS.items()
}

print(f"Execution environment: {EXECUTION_ENVIRONMENT}")
print(f"Package operation: {PACKAGE_OPERATION}")
print(f"Output folder name: {OUTPUT_ROOT.name}")
print(f"Tracking mode: {TRACKING_MODE}")
display_rows(
    [{"resource": key, "version": value} for key, value in PLATFORM_VERSIONS.items()],
    ["resource", "version"],
)
print(f"Notebook SHA-256: {PROVENANCE['notebook_sha256']}")


## Build, verify, and repeat

Local mode builds the package and verifies that a second pass reuses all media. Kaggle mode verifies the private published package, copies it into the run output, and verifies the copy again. Both modes must return identical package hashes.


In [ ]:
if REQUIRE_FRESH_OUTPUT and OUTPUT_ROOT.exists() and any(OUTPUT_ROOT.iterdir()):
    raise RuntimeError("A fresh acceptance run requires an empty output folder.")

if EXECUTION_ENVIRONMENT == "kaggle":
    first_started = time.perf_counter()
    first_audit = verify_model_input_package(PUBLISHED_INPUT_ROOT)
    first_elapsed_seconds = time.perf_counter() - first_started
    shutil.copytree(PUBLISHED_INPUT_ROOT, OUTPUT_ROOT / "model-input")
    second_started = time.perf_counter()
    second_audit = verify_model_input_package(OUTPUT_ROOT / "model-input")
    second_elapsed_seconds = time.perf_counter() - second_started
    first_selection_audit = {
        "cohort_id": SMOKE20_COHORT_ID,
        "membership_sha256": first_audit.membership_sha256,
        "scene_count": 20,
        "asset_group_count": 20,
        "category_count": 20,
        "prefix_coverage_count": 8,
        "stratum_coverage_count": 5,
        "all_checks_passed": True,
        "files_written_this_run": 400,
        "files_reused_this_run": 0,
        "trusted_input_audit": {
            "source_access_mode": "published_opaque_package",
            "source_manifest_sha256": first_audit.manifest_sha256,
            "source_content_checksums_sha256": first_audit.content_checksums_sha256,
            "selected_source_members_verified": first_audit.verified_image_count,
            "all_checks_passed": True,
        },
    }
    second_selection_audit = {
        "files_written_this_run": 0,
        "files_reused_this_run": 400,
    }
else:
    first_started = time.perf_counter()
    first_audit, first_selection_audit = build_smoke20_rgb_package(
        selection_root=SELECTION_ROOT,
        development_root=DEVELOPMENT_ROOT,
        administrative_root=ADMINISTRATIVE_ROOT,
        source_root=SOURCE_ROOT,
        output_root=OUTPUT_ROOT,
        verify_archive_bytes=True,
    )
    first_elapsed_seconds = time.perf_counter() - first_started
    second_started = time.perf_counter()
    second_audit, second_selection_audit = build_smoke20_rgb_package(
        selection_root=SELECTION_ROOT,
        development_root=DEVELOPMENT_ROOT,
        administrative_root=ADMINISTRATIVE_ROOT,
        source_root=SOURCE_ROOT,
        output_root=OUTPUT_ROOT,
        verify_archive_bytes=False,
    )
    second_elapsed_seconds = time.perf_counter() - second_started
verified_audit = verify_model_input_package(OUTPUT_ROOT / "model-input")

if first_audit.to_dict() != second_audit.to_dict():
    raise ValueError("Fresh and resume package audits differ.")
if second_selection_audit["files_reused_this_run"] != 400:
    raise ValueError("The second pass did not reuse all 400 image files.")
if second_selection_audit["files_written_this_run"] != 0:
    raise ValueError("The second pass unexpectedly wrote image files.")
if REQUIRE_FRESH_OUTPUT and first_selection_audit["files_written_this_run"] != 400:
    raise ValueError("The fresh pass did not write all 400 image files.")

resume_summary = {
    "first_files_written": first_selection_audit["files_written_this_run"],
    "first_files_reused": first_selection_audit["files_reused_this_run"],
    "second_files_written": second_selection_audit["files_written_this_run"],
    "second_files_reused": second_selection_audit["files_reused_this_run"],
    "first_elapsed_seconds": round(first_elapsed_seconds, 4),
    "second_elapsed_seconds": round(second_elapsed_seconds, 4),
    "hashes_identical": first_audit.to_dict() == second_audit.to_dict(),
    "package_operation": PACKAGE_OPERATION,
}
display_rows(
    [{"check": key, "value": value} for key, value in resume_summary.items()],
    ["check", "value"],
)


## Package evidence

The following tables and figures describe completeness and media characteristics. The contact sheet is saved only in the private Kaggle or local output bundle. ClearML receives the aggregate charts, not the source images.


In [ ]:
REPORT_ROOT = OUTPUT_ROOT / "report"
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
evidence = collect_model_input_evidence(OUTPUT_ROOT / "model-input")

gate_columns = ["gate", "status", "observed", "expected"]
view_columns = ["view_id", "scene_count", "expected_scene_count", "complete"]
scene_columns = ["scene_index", "opaque_scene_id", "view_count", "expected_view_count", "complete"]
write_csv_rows(REPORT_ROOT / "validation_gates.csv", gate_columns, evidence["gate_rows"])
write_csv_rows(REPORT_ROOT / "view_coverage.csv", view_columns, evidence["view_rows"])
write_csv_rows(REPORT_ROOT / "scene_completeness.csv", scene_columns, evidence["scene_rows"])
(REPORT_ROOT / "media_summary.json").write_text(
    json.dumps(evidence["media_summary"], indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

view_chart = REPORT_ROOT / "view_coverage.png"
size_chart = REPORT_ROOT / "image_size_distribution.png"
contact_sheet = REPORT_ROOT / "private_model_input_contact_sheet.png"
save_bar_chart(
    [row["view_id"] for row in evidence["view_rows"]],
    [row["scene_count"] for row in evidence["view_rows"]],
    view_chart,
    title="Model-input view coverage",
    y_label="Scenes",
)
save_histogram(
    evidence["media_sizes_kib"],
    size_chart,
    title="Model-input image size distribution",
    x_label="Image size (KiB)",
)
save_image_grid(
    evidence["sample_items"],
    contact_sheet,
    title="Private model-input examples",
    columns=5,
)

display_rows(evidence["gate_rows"], gate_columns)
display_rows(evidence["view_rows"], view_columns)
display_rows(
    [{"measure": key, "value": value} for key, value in evidence["media_summary"].items()],
    ["measure", "value"],
)
display(Image.open(view_chart))
display(Image.open(size_chart))
display(Image.open(contact_sheet))


## Run record and ClearML evidence

The run record links the source, notebook, code snapshot, package hashes, two-pass result, tables, and figures. ClearML receives the aggregate gates, tables, scalars, charts, and hash records. It does not receive raw images or private records.


In [ ]:
trusted_input_audit = first_selection_audit["trusted_input_audit"]
safe_selection_summary = {
    key: first_selection_audit[key]
    for key in (
        "cohort_id",
        "membership_sha256",
        "scene_count",
        "asset_group_count",
        "category_count",
        "prefix_coverage_count",
        "stratum_coverage_count",
        "all_checks_passed",
    )
}
tracking_configuration = {
    "scene_count": 20,
    "views_per_scene": 20,
    "expected_image_count": 400,
    "expected_width": 256,
    "expected_height": 256,
    "media_kind": "opaque PNG colour views converted to RGB at model load",
    "fresh_output_required": REQUIRE_FRESH_OUTPUT,
    "package_operation": PACKAGE_OPERATION,
}
configuration_sha256 = sha256(
    json.dumps(tracking_configuration, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()
tracking_properties = {
    "execution_environment": EXECUTION_ENVIRONMENT,
    "release_id": "toysad8k-train-2000-v2",
    "cohort_id": SMOKE20_COHORT_ID,
    "notebook_name": NOTEBOOK_NAME,
    "notebook_sha256": PROVENANCE["notebook_sha256"],
    "git_commit": PROVENANCE["git_commit"],
    "git_dirty": PROVENANCE["git_dirty"],
    "snapshot_sha256": PROVENANCE["snapshot_sha256"],
    "kaggle_notebook_slug": "esreekarreddy/toysad-8k-opaque-package-smoke",
    "input_manifest_sha256": verified_audit.manifest_sha256,
    "configuration_sha256": configuration_sha256,
    **PLATFORM_VERSIONS,
}
tracker = ExperimentTracker.start(
    project_name="Visual Peer Inspector/01 Data Preparation",
    task_name=f"opaque-package-smoke-{EXECUTION_ENVIRONMENT}-candidate",
    task_type="data_processing",
    properties=tracking_properties,
    configuration=tracking_configuration,
)
tracker.add_tags(["candidate", "notebook-03", EXECUTION_ENVIRONMENT, "smoke-20"])

tracking_error = None
try:
    scalar_values = {
        "scenes": verified_audit.scene_count,
        "images": verified_audit.image_count,
        "decoded failures": verified_audit.decoded_image_failure_count,
        "total media MiB": evidence["media_summary"]["total_media_mib"],
        "fresh build seconds": first_elapsed_seconds,
        "resume check seconds": second_elapsed_seconds,
        "fresh images per second": verified_audit.image_count / first_elapsed_seconds,
    }
    for series, value in scalar_values.items():
        tracker.report_scalar(title="Opaque package", series=series, value=value)
    tracker.report_table(
        title="Opaque package",
        series="validation gates",
        columns=gate_columns,
        rows=evidence["gate_rows"],
    )
    tracker.report_table(
        title="Opaque package",
        series="view coverage",
        columns=view_columns,
        rows=evidence["view_rows"],
    )
    tracker.report_table(
        title="Opaque package",
        series="media summary",
        columns=["measure", "value"],
        rows=[{"measure": key, "value": value} for key, value in evidence["media_summary"].items()],
    )
    tracker.report_image_file(
        title="Opaque package",
        series="view coverage chart",
        path=view_chart,
        evidence_kind="aggregate_chart",
    )
    tracker.report_image_file(
        title="Opaque package",
        series="image size distribution",
        path=size_chart,
        evidence_kind="aggregate_chart",
    )
    tracker.upload_summary("trusted input hashes", trusted_input_audit)
    tracker.upload_summary("package audit", verified_audit.to_dict())
    tracker.upload_summary("resume evidence", resume_summary)
    tracker.upload_summary("selection coverage", safe_selection_summary)
    preclose_artifacts = {
        path.relative_to(OUTPUT_ROOT).as_posix(): file_sha256(path)
        for path in sorted(REPORT_ROOT.rglob("*"))
        if path.is_file() and path.name != "artifact_manifest.json"
    }
    tracker.upload_summary("pre-close artifact hashes", preclose_artifacts)
except Exception as error:
    tracking_error = error
    try:
        tracker.mark_failed(type(error).__name__)
    except Exception:
        pass
finally:
    try:
        tracker.close()
    except Exception as error:
        if tracking_error is None:
            tracking_error = error

final_status = "completed" if tracking_error is None else "failed"
run_record = {
    "schema_version": 3,
    "stage": "opaque_package_smoke",
    "status": final_status,
    "tracking_status": final_status,
    "execution_environment": EXECUTION_ENVIRONMENT,
    "notebook_name": NOTEBOOK_NAME,
    "provenance": PROVENANCE,
    "kaggle_notebook_slug": "esreekarreddy/toysad-8k-opaque-package-smoke",
    "platform_versions": PLATFORM_VERSIONS,
    "clearml_task_id": tracker.task_id,
    "configuration": tracking_configuration,
    "configuration_sha256": configuration_sha256,
    "trusted_inputs": trusted_input_audit,
    "package": verified_audit.to_dict(),
    "selection_summary": safe_selection_summary,
    "resume_summary": resume_summary,
    "media_summary": evidence["media_summary"],
    "error_type": type(tracking_error).__name__ if tracking_error else None,
}
run_record_path = REPORT_ROOT / "run_record.json"
run_record_temporary = run_record_path.with_suffix(".json.incomplete")
run_record_temporary.write_text(
    json.dumps(run_record, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
os.replace(run_record_temporary, run_record_path)

artifact_paths = [
    OUTPUT_ROOT / "model-input" / "neutral_corpus_manifest.jsonl",
    OUTPUT_ROOT / "model-input" / "content_checksums.sha256",
    OUTPUT_ROOT / "model-input" / "package_record.json",
]
artifact_paths.extend(
    path for path in sorted(REPORT_ROOT.rglob("*"))
    if path.is_file() and path.name != "artifact_manifest.json"
)
artifact_manifest = {
    path.relative_to(OUTPUT_ROOT).as_posix(): file_sha256(path)
    for path in artifact_paths
}
artifact_manifest_path = REPORT_ROOT / "artifact_manifest.json"
artifact_manifest_path.write_text(
    json.dumps(artifact_manifest, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
from visual_peer_inspector.verification import verify_artifact_manifest

artifact_verification = verify_artifact_manifest(OUTPUT_ROOT)

print(f"Model input: {OUTPUT_ROOT.name}/model-input")
print(f"Run evidence: {OUTPUT_ROOT.name}/report")
print(f"ClearML task ID: {run_record['clearml_task_id'] or 'tracking disabled'}")
print(
    f"Verified anchor/report entries: {artifact_verification['artifact_count']}; "
    f"checksummed media files: {verified_audit.image_count}"
)
if tracking_error is not None:
    raise RuntimeError("ClearML evidence logging failed.") from tracking_error
